In [ ]:
# California Housing 房价预测项目

## 1. 导入库

## 2. 加载数据
from sklearn.datasets import fetch_california_housing
housing = fetch_california_housing()
X = housing.data
y = housing.target
feature_names = housing.feature_names

df = pd.DataFrame(X, columns=feature_names)
df['MedHouseVal'] = y

print(df.head())
print(df.describe())

## 3. 目标变量分布
plt.figure(figsize=(8, 5))
sns.histplot(df['MedHouseVal'], kde=True, bins=50, color='skyblue')
plt.title('California Housing 目标变量分布')
plt.xlabel('房价 (单位: $100,000)')
plt.tight_layout()
plt.savefig('../results/figures/housing_target_dist.png', dpi=150)
plt.show()

## 4. 相关性热力图
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('California Housing 特征相关性热力图')
plt.tight_layout()
plt.savefig('../results/figures/housing_correlation.png', dpi=150)
plt.show()

## 5. 模型训练与评估
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

models = {
    'Linear Regression': LinearRegression(),
    'Ridge (alpha=1.0)': Ridge(alpha=1.0, random_state=42)
}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    print(f"{name}:")
    print(f"  MSE : {mean_squared_error(y_test, y_pred):.4f}")
    print(f"  RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")
    print(f"  MAE : {mean_absolute_error(y_test, y_pred):.4f}")
    print(f"  R²  : {r2_score(y_test, y_pred):.4f}\n")

## 6. 真实值 vs 预测值散点图
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (name, model) in zip(axes, models.items()):
    y_pred = model.predict(X_test_scaled)
    ax.scatter(y_test, y_pred, alpha=0.3, s=10)
    ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
    ax.set_xlabel('真实房价')
    ax.set_ylabel('预测房价')
    ax.set_title(f'{name}\nR² = {r2_score(y_test, y_pred):.4f}')
plt.tight_layout()
plt.savefig('../results/figures/regression_pred_vs_true.png', dpi=150)
plt.show()

## 7. 残差图
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (name, model) in zip(axes, models.items()):
    y_pred = model.predict(X_test_scaled)
    residuals = y_test - y_pred
    ax.scatter(y_pred, residuals, alpha=0.3, s=10)
    ax.axhline(y=0, color='r', linestyle='--')
    ax.set_xlabel('预测值')
    ax.set_ylabel('残差')
    ax.set_title(f'{name} 残差图')
plt.tight_layout()
plt.savefig('../results/figures/residual_plots.png', dpi=150)
plt.show()